<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

And now - introducing the Capstone project:


# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal of today's lab is to make a new python module, `traders.py` that will manage a single trader on our trading floor.

We will experiment and explore in the lab, and then migrate to a python module when we're ready.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Let's start by gathering the MCP params for our trader

In [2]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [3]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [4]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Now create the MCPServerStdio for each

In [5]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [6]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [7]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [8]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Here is the latest news summary on Amazon as of 2025:

1. Amazon Devices & Services 2025 Event:
- Amazon introduced Alexa+, a new enhanced AI assistant service available for $19.99/month but free for Prime members.
- Alexa+ integrates with thousands of services to perform complex tasks like booking reservations, ordering groceries, and more.
- Alexa+ features advanced agentic capabilities to complete actions on behalf of users, and includes an AI Multi-Agent SDK for third-party integrations.
- Alexa+ will roll out first in the US on Echo Show devices and expand in waves over the coming months.

2. Amazon Upfront 2025 Advertising Announcements:
- Amazon unveiled AI-powered streaming ad formats and expanded interactive advertising tools.
- New AI technology enables contextually relevant ads that connect to viewers' current streaming content.
- Advertisers gain advanced tools for precise audience targeting, combining Amazon's first-party data with publisher data while maintaining privacy.
- Introduced solutions span brand awareness to direct conversion metrics across streaming and traditional TV.

3. AWS Updates and Innovations:
- AWS launched Amazon Bedrock AgentCore for secure, scalable deployment of AI agents.
- Announced Amazon Nova customization with SageMaker AI for tailored foundation models.
- New free tier credits and AI training competitions to upskill developers.
- Amazon S3 introduced native vector support for cloud storage, optimized for AI workloads.
- AWS services scaled massively for Prime Day 2025 handling record-breaking volumes.
- AWS recognized as a Leader in 2025 Gartner Magic Quadrant for cloud platforms for the 15th year in a row.
- Celebrated 10 years of Amazon Aurora with milestone innovations.

Overall, Amazon is focusing on expanding AI-driven personal assistant services (Alexa+), innovating in AI-powered advertising, and maintaining leadership with cutting-edge cloud AI and infrastructure innovations via AWS. Prime membership continues to be a major advantage for exclusive offerings.

Would you like me to look into any specific area such as the new Alexa+, Prime Day metrics, or AWS AI innovations in more detail?

### Look at the trace

https://platform.openai.com/traces

In [9]:
ed_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Michael").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Michael")))
display(Markdown(await read_strategy_resource("Michael")))

{"name": "michael", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-09-14 21:59:24", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [10]:
agent_name = "Michael"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [11]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Michael and your account is under your name, Michael.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "michael", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-09-14 21:59:24", 10000.0], ["2025-09-14 22:01:52", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking

### And to run our Trader

In [12]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4.1-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

I researched the current market and found a few high-volatility stocks suitable for aggressive day trading. I focused on Opendoor Technologies (OPEN), Tesla (TSLA), and Nvidia (NVDA) based on their recent volume and price movements.

I bought:
- 100 shares of Opendoor Technologies (OPEN) at around $9.08 per share, aiming to capitalize on its heavy price drop and expected volatility.
- 10 shares of Tesla (TSLA) at around $396.73 per share to take advantage of its strong upward momentum and high trading volume.

These trades align with my aggressive day trading strategy to catch quick price moves. I will monitor these positions closely for opportunities to sell at a profit or cut losses swiftly.

### Then go and look at the trace

http://platform.openai.com/traces


In [13]:
# And let's look at the results of the trading

await read_accounts_resource(agent_name)

'{"name": "michael", "balance": 5123.8672, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"OPEN": 100, "TSLA": 10}, "transactions": [{"symbol": "OPEN", "quantity": 100, "price": 9.088140000000001, "timestamp": "2025-09-14 22:03:18", "rationale": "Taking advantage of the high volatility and large volume drop in Opendoor Technologies for potential quick rebound or further downward play."}, {"symbol": "TSLA", "quantity": 10, "price": 396.73188, "timestamp": "2025-09-14 22:03:20", "rationale": "Buying Tesla shares to capitalize on strong upward momentum and high trading volume for potential short-term gains."}], "portfolio_value_time_series": [["2025-09-14 21:59:24", 10000.0], ["2025-09-14 22:01:52", 10000.0], ["2025-09-14 22:03:18", 9998.186], ["2025-09-14 22:03:20", 9990.2672], ["2025-09-14 22:04:40", 9990.2672]], "total_portfolio_value": 9990.2672, "total_profit_loss": -9.73279999999977}'

### Now it's time to review the Python module made from this:

`mcp_params.py` is where the MCP servers are specified. You'll notice I've brought in some familiar friends: memory and push notifications!

`templates.py` is where the instructions and messages are set up (i.e. the System prompts and User prompts)

`traders.py` brings it all together.

You'll notice I've done something a bit fancy with code like this:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

This is just a tidy way to combine our "with" statements (known as context managers) so that we don't need to do something ugly like this:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

But it's equivalent.


In [14]:
from traders import Trader


In [15]:
trader = Trader("Michael")

In [18]:
await trader.run()

In [17]:
await read_accounts_resource("Michael")

'{"name": "michael", "balance": 1753.2394, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"OPEN": 100, "TSLA": 10, "AMD": 10, "NVDA": 10}, "transactions": [{"symbol": "OPEN", "quantity": 100, "price": 9.088140000000001, "timestamp": "2025-09-14 22:03:18", "rationale": "Taking advantage of the high volatility and large volume drop in Opendoor Technologies for potential quick rebound or further downward play."}, {"symbol": "TSLA", "quantity": 10, "price": 396.73188, "timestamp": "2025-09-14 22:03:20", "rationale": "Buying Tesla shares to capitalize on strong upward momentum and high trading volume for potential short-term gains."}, {"symbol": "AMD", "quantity": 10, "price": 158.88714, "timestamp": "2025-09-14 22:29:56", "rationale": "Strong momentum and volatility in the semiconductor sector make Nvidia a good day trading candidate."}, {"symbol": "NVDA", "quantity": 10, "price": 178.17564, "timestamp": "2025-0

### Now look at the trace

https://platform.openai.com/traces

### How many tools did we use in total?

In [19]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("michael")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 6 MCP servers, and 16 tools


In [ ]:
trader_mcp_servers


In [39]:
from pprint import pprint
for i, params in enumerate(trader_mcp_server_params):
		async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
				tools = await server.list_tools()
				pprint(tools)






[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '